# FD004 DS-A rerun: held-out Regime 0, five seeds, converged training

**Why this notebook exists.** The archived DS-A notebooks (`03_…Heldout_Regime`, `123/03_…FiveSeed`) never saved a model, and
their fixed 25-epoch training stalls for some seeds: the training loss sits at about 28.9, which is exactly the loss of a
model that predicts constant quantiles (28.88 on the DS-A training targets). Seeds 42 and 789 never escaped that plateau, and
456 and 1000 escaped only in the last epochs. The frozen DS-A result (RMSE 24.869) can't be rebuilt from anything archived.

**What stays the same as the frozen pipeline:** data, k-means regimes (seed 42), engine split (174/37/38, seed 42),
Regime 0 endpoint exclusion, training-only sensor scaling, 30-cycle consecutive windows, 14 sensors + 6 one-hot regimes,
20→18 projection, BiLSTM(64) with temporal attention, q10/q50/q90 pinball loss, Adam 1e-3, batch 256, Asymmetric CQR
(α/2 per tail, finite-sample `higher`) and Balanced Engine CQR (within-engine default 0.80 quantile, then the across-engine
finite-sample `higher` quantile), intervals clipped at 0.

**What changes (pre-specified in Part A, before any test result exists):** how long each model trains.
- 20% of the 174 training engines (35 engines, fixed split seed 42) become an **internal validation** set.
  These are training engines, not calibration or test engines. The model is fitted on the other 139.
- Train up to 150 epochs. Early stopping watches internal-validation pinball loss, patience 15, not before epoch 30,
  and restores the best weights.
- **Convergence check (training data only):** a run counts as converged if its best internal-validation loss is at most
  0.60 × the constant-quantile loss on the same validation windows. Every seed is reported either way. No seed is
  replaced or dropped, so nothing is selected on test outcomes.

**Order of operations.** Part A writes the protocol. Part B trains all five seeds, saves every model, and runs the
convergence check, without touching test windows. Part C evaluates the test windows only after Part B's results are
on disk. Don't change Part A after Part C has run.

Run on Colab with a GPU (Runtime → Change runtime type), or locally. For a quick pipeline check, set the environment variable `FD004_SMOKE=1` (1 seed, 3 epochs, separate output folder).

In [1]:
# ---------- Paths and run mode ----------
import os, sys, json, time, random, hashlib, platform
from pathlib import Path

SMOKE_TEST = os.environ.get("FD004_SMOKE") == "1"   # True = 1 seed, 3 epochs, output to *_SMOKE folder (pipeline check only, results meaningless)

def _find_root():
    candidates = [os.environ.get("FD004_ROOT", ""),
                  "/content/drive/MyDrive/FD004_Research",
                  str(Path.home() / "Downloads/FD004_Research")]
    for c in candidates:
        if c and Path(c, "CMAPSSData/train_FD004.txt").exists():
            return Path(c)
    raise FileNotFoundError("Set FD004_ROOT to the FD004_Research folder (it must contain CMAPSSData/train_FD004.txt).")

if "google.colab" in sys.modules and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")

ROOT = _find_root()
DATA_PATH = ROOT / "CMAPSSData/train_FD004.txt"
OUT = Path(os.environ.get("FD004_OUT") or ROOT / ("FD004_DS_A_Rerun_Results_SMOKE" if SMOKE_TEST else "FD004_DS_A_Rerun_Results"))
(OUT / "models").mkdir(parents=True, exist_ok=True)
(OUT / "histories").mkdir(exist_ok=True)
print("Root:", ROOT)
print("Output:", OUT)

Root: <FD004_ROOT>
Output: <FD004_ROOT>/FD004_DS_A_Rerun_Results


## Part A: protocol (written before any training or test evaluation)

In [2]:
PROTOCOL = {
    "experiment": "FD004 DS-A held-out Regime 0, converged five-seed rerun",
    "created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    "data_seed": 42,                      # k-means, engine split, internal-validation split
    "model_seeds": [42] if SMOKE_TEST else [42, 123, 456, 789, 1000],
    "held_out_regime": 0,
    "n_regimes": 6,
    "rul_clip": 125,
    "window_size": 30,
    "settings": ["setting_1", "setting_2", "setting_3"],
    "sensors": ["s_2", "s_3", "s_4", "s_7", "s_8", "s_9", "s_11", "s_12", "s_13", "s_14", "s_15", "s_17", "s_20", "s_21"],
    "split": {"test_size_first": 0.30, "test_size_second": 0.50, "expected_counts": [174, 37, 38]},
    "internal_validation": {"fraction_of_train_engines": 0.20, "split_seed": 42,
                            "note": "training engines only; endpoints filtered exactly like training"},
    "model": {"projection": 18, "bilstm_units": 64, "attention": "FastTemporalAttention",
              "quantiles": [0.10, 0.50, 0.90], "optimizer": "Adam", "learning_rate": 1e-3, "batch_size": 256},
    "training_rule": {"max_epochs": 3 if SMOKE_TEST else 150, "monitor": "val_loss", "patience": 15,
                      "start_from_epoch": 0 if SMOKE_TEST else 30, "restore_best_weights": True},
    "convergence_check": {"rule": "best_val_loss <= ratio * constant_quantile_val_loss", "ratio": 0.60,
                          "constant_quantiles_fitted_on": "fit engines (training engines minus internal validation)",
                          "failed_seeds": "reported and flagged; never replaced or dropped"},
    "calibration": {"alpha": 0.20,
                    "asymmetric": "finite-sample higher quantile at ceil((n+1)(1-alpha/2))/n per tail",
                    "balanced": "within-engine np.quantile(.,0.80) default, then ceil((M+1)(1-alpha))/M higher across engines",
                    "interval_clip_at_zero": True},
    "life_stage": {"Early": "RUL > 80", "Mid": "40 < RUL <= 80", "Late": "RUL <= 40"},
    "bootstrap": {"replicates": 10000, "unit": "test engine", "rng_seed": 42, "ci": [2.5, 97.5]},
    "primary_report": "all five seeds (mean and SD); seed 42 as the single-run reference",
    "secondary_report": "converged seeds only, labelled as such",
}
protocol_path = OUT / "protocol.json"
if protocol_path.exists() and not SMOKE_TEST:
    old = json.loads(protocol_path.read_text())
    changed = {k for k in PROTOCOL if k != "created_utc" and old.get(k) != PROTOCOL[k]}
    assert not changed, f"Protocol already registered and differs in {changed}. Don't edit Part A after a run."
    PROTOCOL["created_utc"] = old["created_utc"]
else:
    protocol_path.write_text(json.dumps(PROTOCOL, indent=2))
print("Protocol sha256:", hashlib.sha256(protocol_path.read_bytes()).hexdigest()[:16])

Protocol sha256: b63e5994f690bcc6


## Data, regimes, split and windows (identical to the frozen DS-A pipeline)

In [3]:
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

P = PROTOCOL
SET, SENS, W, HO = P["settings"], P["sensors"], P["window_size"], P["held_out_regime"]
cols = ["unit_id", "time_cycles"] + SET + [f"s_{i}" for i in range(1, 22)]

raw = pd.read_csv(DATA_PATH, sep=r"\s+", header=None).iloc[:, :len(cols)]
raw.columns = cols
raw["RUL"] = (raw.groupby("unit_id")["time_cycles"].transform("max") - raw["time_cycles"]).clip(upper=P["rul_clip"])

setting_scaler = StandardScaler().fit(raw[SET])
kmeans = KMeans(n_clusters=P["n_regimes"], random_state=P["data_seed"], n_init=10)
raw["regime"] = kmeans.fit_predict(setting_scaler.transform(raw[SET]))
assert len(raw) == 61249
print("Regime counts:", raw["regime"].value_counts().sort_index().to_dict())
assert raw["regime"].value_counts()[HO] == 15395, "Regime 0 size differs from the frozen run"

engines = np.array(sorted(raw["unit_id"].unique()))
train_engines, temp = train_test_split(engines, test_size=0.30, random_state=P["data_seed"])
cal_engines, test_engines = train_test_split(temp, test_size=0.50, random_state=P["data_seed"])
assert [len(train_engines), len(cal_engines), len(test_engines)] == P["split"]["expected_counts"]

# Same split code as DS-B; check it against the frozen DS-B split file when available.
dsb_split = ROOT / "FD004_DS_B_Results/DS_B_Engine_Split.csv"
if dsb_split.exists():
    s = pd.read_csv(dsb_split)
    assert set(s.loc[s.split == "test", "unit_id"]) == set(test_engines), "Test engines differ from the frozen split"
    assert set(s.loc[s.split == "calibration", "unit_id"]) == set(cal_engines)
    print("Engine split matches the frozen split file.")

fit_engines, ival_engines = train_test_split(train_engines, test_size=P["internal_validation"]["fraction_of_train_engines"],
                                             random_state=P["internal_validation"]["split_seed"])
print(f"Train engines {len(train_engines)} -> fit {len(fit_engines)} + internal validation {len(ival_engines)}")

# Sensor scaler: all 174 training engines, non-held-out rows (as in the frozen DS-A notebook).
sensor_scaler = StandardScaler().fit(raw.loc[raw.unit_id.isin(train_engines) & (raw.regime != HO), SENS])
traj = raw.copy()
traj[SENS] = sensor_scaler.transform(traj[SENS])
traj = traj.sort_values(["unit_id", "time_cycles"]).reset_index(drop=True)

def make_windows(engine_ids, mode):
    X, y, ids, cyc = [], [], [], []
    eye = np.eye(P["n_regimes"], dtype=np.float32)
    for e in sorted(engine_ids):
        g = traj[traj.unit_id == e]
        sens = g[SENS].to_numpy(np.float32); reg = g["regime"].to_numpy()
        rul = g["RUL"].to_numpy(np.float32); t = g["time_cycles"].to_numpy()
        for end in range(W - 1, len(g)):
            if (reg[end] == HO) != (mode == "heldout"):
                continue
            sl = slice(end - W + 1, end + 1)
            assert np.all(np.diff(t[sl]) == 1)
            X.append(np.concatenate([sens[sl], eye[reg[sl]]], axis=-1))
            y.append(rul[end]); ids.append(e); cyc.append(t[end])
    return np.asarray(X), np.asarray(y, np.float32), np.asarray(ids), np.asarray(cyc)

X_fit, y_fit, id_fit, _ = make_windows(fit_engines, "seen")
X_iv, y_iv, id_iv, _ = make_windows(ival_engines, "seen")
X_cal, y_cal, id_cal, cyc_cal = make_windows(cal_engines, "seen")
X_test, y_test, id_test, cyc_test = make_windows(test_engines, "heldout")   # not used until Part C

assert len(y_fit) + len(y_iv) == 28322 and len(y_cal) == 6021 and len(y_test) == 2043, "Window counts differ from frozen"
print("fit", X_fit.shape, "| internal val", X_iv.shape, "| cal", X_cal.shape, "| test", X_test.shape)

np.savez(OUT / "preprocessing.npz",
         kmeans_centers=kmeans.cluster_centers_, setting_mean=setting_scaler.mean_, setting_scale=setting_scaler.scale_,
         sensor_mean=sensor_scaler.mean_, sensor_scale=sensor_scaler.scale_)
pd.concat([pd.DataFrame({"split": n, "unit_id": v}) for n, v in
           [("fit", fit_engines), ("internal_val", ival_engines), ("calibration", cal_engines), ("test", test_engines)]]
          ).to_csv(OUT / "engine_split.csv", index=False)

Regime counts: {0: 15395, 1: 9224, 2: 9139, 3: 9091, 4: 9238, 5: 9162}
Engine split matches the frozen split file.
Train engines 174 -> fit 139 + internal validation 35


fit (22569, 30, 20) | internal val (5753, 30, 20) | cal (6021, 30, 20) | test (2043, 30, 20)


## Part B: train five seeds and check convergence (no test data)

In [4]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, LSTM, Bidirectional, Layer
from tensorflow.keras.models import Model

Q = P["model"]["quantiles"]

@tf.keras.utils.register_keras_serializable(package="fd004")
class FastTemporalAttention(Layer):
    def build(self, input_shape):
        d = input_shape[-1]
        self.W = self.add_weight(name="W", shape=(d, d), initializer="glorot_uniform")
        self.b = self.add_weight(name="b", shape=(d,), initializer="zeros")
        self.v = self.add_weight(name="v", shape=(d, 1), initializer="glorot_uniform")
        super().build(input_shape)
    def call(self, x):
        e = tf.nn.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        w = tf.nn.softmax(tf.tensordot(e, self.v, axes=1), axis=1)
        return tf.reduce_sum(x * w, axis=1)

@tf.keras.utils.register_keras_serializable(package="fd004")
def multi_quantile_pinball_loss(y_true, y_pred):
    y_true = tf.cast(tf.reshape(y_true, (-1, 1)), tf.float32)
    losses = [tf.maximum(q * (y_true - y_pred[:, i:i+1]), (q - 1.0) * (y_true - y_pred[:, i:i+1])) for i, q in enumerate(Q)]
    return tf.reduce_mean(tf.add_n(losses))

def build_model():
    m = P["model"]
    inp = Input(shape=(W, 20), name="input_window")
    h = Dense(m["projection"], use_bias=False, name="projection")(inp)
    h = Bidirectional(LSTM(m["bilstm_units"], return_sequences=True), name="bilstm")(h)
    h = FastTemporalAttention(name="temporal_attention")(h)
    out = Dense(3, activation="linear", name="quantiles")(h)
    model = Model(inp, out, name="FD004_DS_A_rerun")
    model.compile(optimizer=tf.keras.optimizers.Adam(m["learning_rate"]), loss=multi_quantile_pinball_loss)
    return model

def set_seed(s):
    random.seed(s); np.random.seed(s); tf.random.set_seed(s); os.environ["PYTHONHASHSEED"] = str(s)

def constant_quantile_loss(y_ref, y_eval):
    return float(sum(np.mean(np.maximum(q * (y_eval - np.quantile(y_ref, q)), (q - 1) * (y_eval - np.quantile(y_ref, q))))
                     for q in Q))

const_iv = constant_quantile_loss(y_fit, y_iv)
print(f"Constant-quantile loss on internal validation: {const_iv:.3f}  -> convergence threshold "
      f"{P['convergence_check']['ratio'] * const_iv:.3f}")

env = {"python": platform.python_version(), "platform": platform.platform(), "tensorflow": tf.__version__,
       "numpy": np.__version__, "pandas": pd.__version__,
       "gpu": [d.name for d in tf.config.list_physical_devices("GPU")]}
(OUT / "environment.json").write_text(json.dumps(env, indent=2))
print(env)

Constant-quantile loss on internal validation: 29.081  -> convergence threshold 17.449
{'python': '3.12.13', 'platform': 'macOS-26.5.2-arm64-arm-64bit', 'tensorflow': '2.20.0', 'numpy': '2.0.2', 'pandas': '2.2.3', 'gpu': []}


In [5]:
tr = P["training_rule"]
train_log = []
for seed in P["model_seeds"]:
    tf.keras.backend.clear_session()
    set_seed(seed)
    model = build_model()
    stop = tf.keras.callbacks.EarlyStopping(monitor=tr["monitor"], patience=tr["patience"],
                                            start_from_epoch=tr["start_from_epoch"], restore_best_weights=tr["restore_best_weights"])
    t0 = time.time()
    hist = model.fit(X_fit, y_fit, validation_data=(X_iv, y_iv), epochs=tr["max_epochs"],
                     batch_size=P["model"]["batch_size"], shuffle=True, callbacks=[stop], verbose=2)
    h = pd.DataFrame(hist.history); h.insert(0, "epoch", np.arange(1, len(h) + 1))
    h.to_csv(OUT / "histories" / f"seed{seed}_history.csv", index=False)
    model.save(OUT / "models" / f"DS_A_rerun_seed{seed}.keras")
    best = float(h["val_loss"].min())
    train_log.append({"seed": seed, "epochs_run": len(h), "best_epoch": int(h.loc[h.val_loss.idxmin(), "epoch"]),
                      "best_val_loss": best, "final_train_loss": float(h["loss"].iloc[-1]),
                      "constant_val_loss": const_iv, "ratio": best / const_iv,
                      "converged": best <= P["convergence_check"]["ratio"] * const_iv,
                      "minutes": (time.time() - t0) / 60})
    print(train_log[-1])

train_log = pd.DataFrame(train_log)
train_log.to_csv(OUT / "convergence_check.csv", index=False)
display(train_log.round(3))
if not train_log.converged.all():
    print("WARNING: some seeds did not converge. They stay in the primary report, flagged. Don't rerun with new seeds.")

Epoch 1/150


89/89 - 5s - 60ms/step - loss: 118.2158 - val_loss: 103.8230


Epoch 2/150


89/89 - 4s - 50ms/step - loss: 96.7650 - val_loss: 88.9148


Epoch 3/150


...

89/89 - 7s - 79ms/step - loss: 11.5755 - val_loss: 12.7596


Epoch 53/150


89/89 - 7s - 84ms/step - loss: 11.5395 - val_loss: 12.8915


{'seed': 1000, 'epochs_run': 53, 'best_epoch': 38, 'best_val_loss': 11.271327018737793, 'final_train_loss': 11.5394926071167, 'constant_val_loss': 29.081470489501953, 'ratio': 0.3875776165722638, 'converged': True, 'minutes': 6.285669811566671}


,seed,epochs_run,best_epoch,best_val_loss,final_train_loss,constant_val_loss,ratio,converged,minutes
0,42,86,71,11.079,10.107,29.081,0.381,True,10.852
1,123,90,75,10.604,9.965,29.081,0.365,True,10.861
2,456,60,45,13.313,11.546,29.081,0.458,True,7.039
3,789,84,69,15.167,11.276,29.081,0.522,True,10.001
4,1000,53,38,11.271,11.539,29.081,0.388,True,6.286


## Part C: evaluation on the held-out Regime 0 test windows (run only after Part B is saved)

In [6]:
assert (OUT / "convergence_check.csv").exists(), "Run Part B first."

def cq(scores, a):
    n = len(scores)
    return float(np.quantile(scores, min(1.0, np.ceil((n + 1) * (1 - a)) / n), method="higher"))

def stage_of(r):
    return np.where(r > 80, "Early", np.where(r > 40, "Mid", "Late"))

def winkler(y, lo, hi, a):
    return (hi - lo) + (2 / a) * (lo - y) * (y < lo) + (2 / a) * (y - hi) * (y > hi)

A = P["calibration"]["alpha"]
rows, engine_rows, stage_rows, global_rows = [], [], [], []
for seed in P["model_seeds"]:
    model = tf.keras.models.load_model(OUT / "models" / f"DS_A_rerun_seed{seed}.keras")
    pc = model.predict(X_cal, batch_size=512, verbose=0)
    pt = model.predict(X_test, batch_size=512, verbose=0)

    ql, qu = cq(pc[:, 0] - y_cal, A / 2), cq(y_cal - pc[:, 2], A / 2)
    err = np.maximum(pc[:, 0] - y_cal, y_cal - pc[:, 2])
    eng_q80 = np.array([np.quantile(err[id_cal == e], 0.80) for e in np.unique(id_cal)])
    M = len(eng_q80)
    qb = float(np.quantile(eng_q80, min(1.0, np.ceil((M + 1) * (1 - A)) / M), method="higher"))

    df = pd.DataFrame({"seed": seed, "engine_id": id_test, "cycle": cyc_test, "true_RUL": y_test,
                       "q10": pt[:, 0], "q50": pt[:, 1], "q90": pt[:, 2], "stage": stage_of(y_test)})
    df["asym_lower"] = np.maximum(0, df.q10 - ql); df["asym_upper"] = np.maximum(0, df.q90 + qu)
    df["bal_lower"] = np.maximum(0, df.q10 - qb); df["bal_upper"] = np.maximum(0, df.q90 + qb)
    for m in ["asym", "bal"]:
        df[f"{m}_covered"] = (df.true_RUL >= df[f"{m}_lower"]) & (df.true_RUL <= df[f"{m}_upper"])
        df[f"{m}_winkler"] = winkler(df.true_RUL, df[f"{m}_lower"], df[f"{m}_upper"], A)
    df["error_q50_minus_true"] = df.q50 - df.true_RUL
    rows.append(df)
    pd.DataFrame({"seed": seed, "engine_id": id_cal, "cycle": cyc_cal, "true_RUL": y_cal,
                  "q10": pc[:, 0], "q50": pc[:, 1], "q90": pc[:, 2]}
                 ).to_csv(OUT / f"calibration_predictions_seed{seed}.csv", index=False)

    rmse = float(np.sqrt(np.mean(df.error_q50_minus_true ** 2)))
    for m, name in [("asym", "Asymmetric CQR"), ("bal", "Balanced Engine CQR")]:
        global_rows.append({"seed": seed, "method": name, "RMSE": rmse, "MAE": float(df.error_q50_minus_true.abs().mean()),
                            "PICP": 100 * df[f"{m}_covered"].mean(), "MPIW": float((df[f"{m}_upper"] - df[f"{m}_lower"]).mean()),
                            "Winkler": float(df[f"{m}_winkler"].mean()),
                            "Q_lower": ql if m == "asym" else qb, "Q_upper": qu if m == "asym" else qb})
    for e, g in df.groupby("engine_id"):
        engine_rows.append({"seed": seed, "engine_id": e, "n_windows": len(g),
                            "RMSE": float(np.sqrt(np.mean(g.error_q50_minus_true ** 2))), "MAE": float(g.error_q50_minus_true.abs().mean()),
                            "bias": float(g.error_q50_minus_true.mean()),
                            "Asym_Coverage": 100 * g.asym_covered.mean(), "Bal_Coverage": 100 * g.bal_covered.mean(),
                            "Asym_Width": float((g.asym_upper - g.asym_lower).mean()), "Bal_Width": float((g.bal_upper - g.bal_lower).mean())})
    for s, g in df.groupby("stage"):
        stage_rows.append({"seed": seed, "stage": s, "N": len(g), "RMSE": float(np.sqrt(np.mean(g.error_q50_minus_true ** 2))),
                           "MAE": float(g.error_q50_minus_true.abs().mean()), "signed_bias": float(g.error_q50_minus_true.mean()),
                           "Asym_Coverage": 100 * g.asym_covered.mean(), "Bal_Coverage": 100 * g.bal_covered.mean()})

pred = pd.concat(rows, ignore_index=True)
pred.to_csv(OUT / "test_predictions_rowlevel.csv", index=False)
glob = pd.DataFrame(global_rows).merge(train_log[["seed", "converged"]], on="seed")
per_engine = pd.DataFrame(engine_rows)
stages = pd.DataFrame(stage_rows)
glob.to_csv(OUT / "global_by_seed.csv", index=False)
per_engine.to_csv(OUT / "per_engine_by_seed.csv", index=False)
stages.to_csv(OUT / "life_stage_by_seed.csv", index=False)
display(glob.round(3))

,seed,method,RMSE,MAE,PICP,MPIW,Winkler,Q_lower,Q_upper,converged
0,42,Asymmetric CQR,24.097,16.216,73.568,45.237,74.496,1.291,0.074,True
1,42,Balanced Engine CQR,24.097,16.216,91.043,68.851,80.311,12.866,12.866,True
2,123,Asymmetric CQR,22.905,14.752,72.051,37.448,72.948,1.116,0.199,True
3,123,Balanced Engine CQR,22.905,14.752,88.546,60.449,75.358,12.606,12.606,True
4,456,Asymmetric CQR,25.316,16.209,76.016,50.325,80.190,1.540,1.747,True
5,456,Balanced Engine CQR,25.316,16.209,90.553,73.110,85.844,13.660,13.660,True
6,789,Asymmetric CQR,27.544,18.208,78.463,56.803,91.649,4.997,7.608,True
7,789,Balanced Engine CQR,27.544,18.208,90.798,83.129,96.530,20.748,20.748,True
8,1000,Asymmetric CQR,25.158,15.501,73.470,43.875,79.978,-1.547,2.338,True
9,1000,Balanced Engine CQR,25.158,15.501,88.350,65.389,81.632,11.668,11.668,True


In [7]:
# Engine-level summaries (Table 5 style), RMSE-coverage correlation, problem engines
from scipy.stats import pearsonr, spearmanr

summ = []
for (seed, m), col in [((s, m), c) for s in P["model_seeds"] for m, c in [("Asymmetric CQR", "Asym_Coverage"), ("Balanced Engine CQR", "Bal_Coverage")]]:
    v = per_engine.loc[per_engine.seed == seed, col].to_numpy()
    summ.append({"seed": seed, "method": m, "mean": v.mean(), "median": np.median(v), "p10": np.percentile(v, 10),
                 "p25": np.percentile(v, 25), "worst": v.min(), "below80": int((v < 80).sum()), "below70": int((v < 70).sum())})
summ = pd.DataFrame(summ)
summ.to_csv(OUT / "engine_level_summary_by_seed.csv", index=False)

corr = []
for seed, g in per_engine.groupby("seed"):
    r, pr = pearsonr(g.RMSE, g.Bal_Coverage); rho, ps = spearmanr(g.RMSE, g.Bal_Coverage)
    corr.append({"seed": seed, "pearson_r": r, "pearson_p": pr, "spearman_rho": rho, "spearman_p": ps,
                 "problem_engines_bal_below80": " ".join(map(str, sorted(g.loc[g.Bal_Coverage < 80, "engine_id"])))})
corr = pd.DataFrame(corr)
corr.to_csv(OUT / "rmse_coverage_correlation_by_seed.csv", index=False)
display(summ.round(2)); display(corr.round(3))

,seed,method,mean,median,p10,p25,worst,below80,below70
0,42,Asymmetric CQR,69.00,73.57,37.60,53.68,18.52,23,17
1,42,Balanced Engine CQR,88.07,93.46,65.71,84.58,40.74,8,6
2,123,Asymmetric CQR,69.43,66.29,40.76,47.89,24.00,20,20
3,123,Balanced Engine CQR,86.09,94.82,59.69,78.42,50.00,11,8
4,456,Asymmetric CQR,72.51,77.60,42.49,58.56,28.00,19,18
5,456,Balanced Engine CQR,88.03,92.72,70.26,79.38,48.00,11,3
6,789,Asymmetric CQR,74.66,81.17,46.38,66.38,22.22,18,12
7,789,Balanced Engine CQR,88.77,95.39,68.97,84.09,40.00,6,4
8,1000,Asymmetric CQR,69.44,73.76,36.63,55.22,28.00,23,18
9,1000,Balanced Engine CQR,86.12,89.59,64.77,81.43,44.00,9,5


,seed,pearson_r,pearson_p,spearman_rho,spearman_p,problem_engines_bal_below80
0,42,-0.947,0.0,-0.962,0.0,17 105 115 156 184 194 240 245
1,123,-0.954,0.0,-0.952,0.0,17 26 105 115 120 156 181 184 194 240 245
2,456,-0.957,0.0,-0.846,0.0,17 70 85 105 115 120 156 184 194 240 245
3,789,-0.924,0.0,-0.856,0.0,17 105 115 156 184 240
4,1000,-0.920,0.0,-0.866,0.0,17 85 105 115 156 184 194 240 245


In [8]:
# Engine-cluster bootstrap (10,000 replicates) for global PICP/MPIW and the paired Balanced - Asymmetric difference
B = P["bootstrap"]
boot_rows = []
for seed in P["model_seeds"]:
    g = pred[pred.seed == seed]
    agg = g.groupby("engine_id").agg(n=("true_RUL", "size"), ca=("asym_covered", "sum"), cb=("bal_covered", "sum"),
                                     wa=("asym_upper", "sum"), wb=("bal_upper", "sum"),
                                     la=("asym_lower", "sum"), lb=("bal_lower", "sum"))
    n, ca, cb = agg.n.to_numpy(), agg.ca.to_numpy(), agg.cb.to_numpy()
    wa, wb = (agg.wa - agg.la).to_numpy(), (agg.wb - agg.lb).to_numpy()
    rng = np.random.default_rng(B["rng_seed"])
    idx = rng.integers(0, len(n), size=(B["replicates"], len(n)))
    N = n[idx].sum(1)
    stats = {"asym_picp": 100 * ca[idx].sum(1) / N, "bal_picp": 100 * cb[idx].sum(1) / N,
             "asym_mpiw": wa[idx].sum(1) / N, "bal_mpiw": wb[idx].sum(1) / N}
    stats["bal_minus_asym_picp"] = stats["bal_picp"] - stats["asym_picp"]
    point = {"asym_picp": 100 * ca.sum() / n.sum(), "bal_picp": 100 * cb.sum() / n.sum(),
             "asym_mpiw": wa.sum() / n.sum(), "bal_mpiw": wb.sum() / n.sum()}
    point["bal_minus_asym_picp"] = point["bal_picp"] - point["asym_picp"]
    for k, v in stats.items():
        lo, hi = np.percentile(v, B["ci"])
        boot_rows.append({"seed": seed, "stat": k, "point": point[k], "ci_low": lo, "ci_high": hi})
boot = pd.DataFrame(boot_rows)
boot.to_csv(OUT / "bootstrap_by_seed.csv", index=False)
display(boot.round(2))

,seed,stat,point,ci_low,ci_high
0,42,asym_picp,73.57,67.03,79.56
1,42,bal_picp,91.04,86.88,94.39
2,42,asym_mpiw,45.24,43.55,47.04
3,42,bal_mpiw,68.85,67.19,70.64
4,42,bal_minus_asym_picp,17.47,13.86,21.33
5,123,asym_picp,72.05,64.76,79.00
6,123,bal_picp,88.55,84.22,92.42
7,123,asym_mpiw,37.45,36.00,39.03
8,123,bal_mpiw,60.45,59.06,61.97
9,123,bal_minus_asym_picp,16.50,12.69,20.47


In [9]:
# Five-seed summary: all seeds (primary) and converged seeds only (secondary)
def agg(df, label):
    out = df.groupby("method")[["PICP", "MPIW", "Winkler", "RMSE", "MAE"]].agg(["mean", "std"])
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    out.insert(0, "n_seeds", df.groupby("method").size())
    out.insert(0, "seeds", label)
    return out.reset_index()

five = pd.concat([agg(glob, "all"), agg(glob[glob.converged], "converged only")], ignore_index=True)
five.to_csv(OUT / "five_seed_summary.csv", index=False)
display(five.round(3))

stage_summary = stages.groupby("stage")[["RMSE", "signed_bias", "Bal_Coverage", "Asym_Coverage"]].agg(["mean", "std"]).round(2)
stage_summary.to_csv(OUT / "life_stage_five_seed_summary.csv")
display(stage_summary)

frozen = {"RMSE": 24.869, "Asym PICP": 73.13, "Bal PICP": 90.21}
s42 = glob[glob.seed == 42].set_index("method")
if len(s42):
    print("Seed 42 vs archived frozen summary (for reference only, not a gate):")
    print(f"  RMSE {s42.RMSE.iloc[0]:.3f} vs {frozen['RMSE']} | Asym {s42.loc['Asymmetric CQR','PICP']:.2f} vs {frozen['Asym PICP']}"
          f" | Balanced {s42.loc['Balanced Engine CQR','PICP']:.2f} vs {frozen['Bal PICP']}")

manifest = {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in sorted(OUT.rglob("*")) if p.is_file()}
(OUT / "manifest_sha256.json").write_text(json.dumps(manifest, indent=2))
print(f"Done. {len(manifest)} files hashed in", OUT)

,method,seeds,n_seeds,PICP_mean,PICP_std,MPIW_mean,MPIW_std,Winkler_mean,Winkler_std,RMSE_mean,RMSE_std,MAE_mean,MAE_std
0,Asymmetric CQR,all,5,74.714,2.535,46.738,7.262,79.852,7.343,25.004,1.718,16.177,1.286
1,Balanced Engine CQR,all,5,89.858,1.300,70.186,8.598,83.935,7.973,25.004,1.718,16.177,1.286
2,Asymmetric CQR,converged only,5,74.714,2.535,46.738,7.262,79.852,7.343,25.004,1.718,16.177,1.286
3,Balanced Engine CQR,converged only,5,89.858,1.300,70.186,8.598,83.935,7.973,25.004,1.718,16.177,1.286


RMSE       signed_bias       Bal_Coverage       Asym_Coverage      
        mean   std        mean   std         mean   std          mean   std
stage                                                                      
Early  18.33  1.31       -3.05  2.54        94.43  1.33         83.52  2.22
Late   25.23  4.75       11.05  4.85        89.43  1.87         70.37  5.88
Mid    39.11  3.12       17.64  2.73        75.52  6.36         50.72  6.03

Seed 42 vs archived frozen summary (for reference only, not a gate):
  RMSE 24.097 vs 24.869 | Asym 73.57 vs 73.13 | Balanced 91.04 vs 90.21
Done. 29 files hashed in <FD004_ROOT>/FD004_DS_A_Rerun_Results
